# Silver Layer — Transaction Data

**Runs in two modes automatically:**

| Mode | Bronze reads from | Silver writes to |
|------|------------------|-----------------|
| **Google Colab** | `MyDrive/FinPulse/data/bronze/transactions` | `MyDrive/FinPulse/data/silver/transactions` |
| **Local (VS Code)** | `<project_root>/data/bronze/transactions` | `<project_root>/data/silver/transactions` |

Override any path via environment variables: `BRONZE_TRANSACTIONS_PATH`, `SILVER_OUTPUT_PATH`, `ICEBERG_WAREHOUSE`, `GDRIVE_FINPULSE_ROOT`.

In [ ]:
import sys
print("Python version:", sys.version)
print("Setup verified ✅")

: 

In [ ]:
# Dependencies should be managed via requirements.txt or Docker environment
# e.g. pip install pyspark pyiceberg pandas python-dotenv

In [ ]:
# ── Verify Bronze data exists locally ────────────────────────
import os
from pathlib import Path

def _find_project_root(marker: str = "pyproject.toml") -> Path:
    current = Path(os.path.abspath("")).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists():
            return candidate
    return current.parent  # fallback: one level above notebooks/

PROJECT_ROOT = _find_project_root()
BRONZE_BASE = str(PROJECT_ROOT / "data" / "bronze" / "transactions")

# Find latest partition
partitions = [d for d in os.listdir(BRONZE_BASE) if d.startswith("date=")]
latest = sorted(partitions)[-1]
BRONZE_FILE = os.path.join(BRONZE_BASE, latest, "transactions_raw.csv")

# Verify file exists
if os.path.exists(BRONZE_FILE):
    size_mb = os.path.getsize(BRONZE_FILE) / (1024 * 1024)
    print(f"✅ Bronze file found: {BRONZE_FILE}")
    print(f"   Size: {size_mb:.1f} MB")
    print(f"   Partition: {latest}")
else:
    raise FileNotFoundError(f"❌ Bronze file not found at: {BRONZE_FILE}")

os.environ["BRONZE_TRANSACTIONS_PATH"] = BRONZE_BASE
print(f"\n✅ Bronze path configured: {BRONZE_BASE}")

In [ ]:
# ============================================================
# FinPulse - Silver Layer: Transaction Data
# Purpose: Clean, validate and enrich Bronze transaction data
# Author: Amanjot Kaur
# ============================================================
import os
import sys
from pathlib import Path
from datetime import datetime

# -- STEP 1: Robust environment setup ---------------------------
def _find_project_root(marker: str = 'pyproject.toml') -> Path:
    current = Path(os.getcwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists(): return candidate
    return current

PROJECT_ROOT = _find_project_root()

# Force clear invalid SPARK_HOME and set Python environment
if 'SPARK_HOME' in os.environ and not Path(os.environ['SPARK_HOME']).is_dir():
    del os.environ['SPARK_HOME']
    print('OK: Invalid SPARK_HOME cleared.')

# Configure Java and Hadoop - Using Project-Local Hadoop Binaries
os.environ['JAVA_HOME'] = r'C:\java\jdk-21'
os.environ['HADOOP_HOME'] = str(PROJECT_ROOT / 'hadoop')
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Inject bins into PATH session-wide to ensure DLLs (winutils/hadoop) are found
hadoop_bin = Path(os.environ['HADOOP_HOME']) / 'bin'
java_bin = Path(os.environ['JAVA_HOME']) / 'bin'

for p in [java_bin, hadoop_bin]:
    if p.is_dir() and str(p) not in os.environ['PATH']:
        os.environ['PATH'] = str(p) + os.pathsep + os.environ['PATH']

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType
from dotenv import load_dotenv
load_dotenv(override=True)

print(f'HADOOP_HOME set to: {os.environ["HADOOP_HOME"]}')
print('Environment setup successful')

# -- Path configuration -------------------------------------
def _is_colab() -> bool:
    try: return 'google.colab' in sys.modules or os.path.exists('/content')
    except: return False

ON_COLAB = _is_colab()
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    _GDRIVE_ROOT = Path(os.environ.get('GDRIVE_FINPULSE_ROOT', '/content/drive/MyDrive/FinPulse'))
    _DEFAULT_BRONZE = str(_GDRIVE_ROOT / 'data' / 'bronze' / 'transactions')
    _DEFAULT_SILVER = str(_GDRIVE_ROOT / 'data' / 'silver' / 'transactions')
    print('Mode: Google Colab')
else:
    # Local Detection: Prioritize the G:\ Google Drive mount
    _GDRIVE_ROOT = Path(os.environ.get('GDRIVE_FINPULSE_ROOT', 'G:/My Drive/FinPulse'))
    
    if _GDRIVE_ROOT.exists():
        _DEFAULT_BRONZE = str(_GDRIVE_ROOT / 'data' / 'bronze' / 'transactions')
        _DEFAULT_SILVER = str(_GDRIVE_ROOT / 'data' / 'silver' / 'transactions')
        print(f'Mode: Hybrid (G-Drive Connected) | Root: {_GDRIVE_ROOT}')
    else:
        _PROJECT_ROOT = _find_project_root()
        _DEFAULT_BRONZE = str(_PROJECT_ROOT / 'data' / 'bronze' / 'transactions')
        _DEFAULT_SILVER = str(_PROJECT_ROOT / 'data' / 'silver' / 'transactions')
        print(f'Mode: Local | Root: {_PROJECT_ROOT}')

BRONZE_PATH = os.environ.get('BRONZE_TRANSACTIONS_PATH', _DEFAULT_BRONZE)
SILVER_OUTPUT_PATH = os.environ.get('SILVER_OUTPUT_PATH', _DEFAULT_SILVER)

PIPELINE_VERSION = 'v1.0'
INGESTION_TIMESTAMP = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

spark = (
    SparkSession.builder
    .appName('FinPulse-Silver-Transactions')
    .master(os.environ.get('SPARK_MASTER', 'local[*]'))
    .config('spark.driver.memory', '2g')
    .config('spark.executor.memory', '2g')
    # ULTRA FIX: Force Java to bypass all Hadoop Native IO checks on Windows
    .config('spark.driver.extraJavaOptions', '-Dorg.apache.hadoop.io.nativeio.NativeIO.Windows.should_use_native_io=false')
    .config('spark.executor.extraJavaOptions', '-Dorg.apache.hadoop.io.nativeio.NativeIO.Windows.should_use_native_io=false')
    # WORKAROUND: Use RawLocalFileSystem to bypass missing hadoop.dll native checks
    .config('spark.hadoop.fs.file.impl', 'org.apache.hadoop.fs.RawLocalFileSystem')
    .config('spark.hadoop.fs.file.impl.disable.cache', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print(f'Spark session ready | version {spark.version}')


In [ ]:
def load_bronze_transactions(bronze_path: str):
    """
    Loads Bronze transaction data — handles all three storage layouts:

      Layout 1 (partitioned folder):  bronze_path/date=YYYY-MM-DD/transactions_raw.csv
      Layout 2 (flat folder):         bronze_path/transactions_raw.csv  (or any single .csv)
      Layout 3 (direct file path):    bronze_path ends with .csv
    """
    print("=" * 60)
    print("LOADING BRONZE TRANSACTION DATA")
    print("=" * 60)
    print(f"Source: {bronze_path}")

    p = Path(bronze_path)

    # ── Layout 3: path IS a CSV file ──────────────────────────
    if str(bronze_path).endswith(".csv"):
        if not p.exists():
            raise FileNotFoundError(f"CSV not found: {bronze_path}")
        file_path = str(p)
        print(f"Layout: direct CSV file")

    # ── Layout 1: partitioned by date= folders ─────────────────
    elif any(d.startswith("date=") for d in os.listdir(bronze_path)):
        partitions = sorted(d for d in os.listdir(bronze_path) if d.startswith("date="))
        latest = partitions[-1]
        file_path = str(p / latest / "transactions_raw.csv")
        print(f"Layout: partitioned  |  loading latest partition: {latest}")

    # ── Layout 2: flat folder with CSV inside ──────────────────
    else:
        csvs = sorted(p.glob("*.csv"))
        if not csvs:
            raise FileNotFoundError(
                f"No CSV files found in: {bronze_path}\n"
                f"Set BRONZE_TRANSACTIONS_PATH to the correct folder or file."
            )
        file_path = str(csvs[-1])  # pick alphabetically latest
        print(f"Layout: flat folder  |  loading: {Path(file_path).name}")

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(file_path)
    )

    print(f"Records loaded : {df.count():,}")
    print(f"Columns        : {df.columns}")
    return df

bronze_df = load_bronze_transactions(BRONZE_PATH)
bronze_df.printSchema()

In [ ]:
def apply_silver_transformations(df):
    """Applies all Silver layer transformations."""
    print("=" * 60)
    print("APPLYING SILVER TRANSFORMATIONS")
    print("=" * 60)

    # Step 1: Remove zero amount transactions
    initial_count = df.count()
    df = df.filter(F.col("amount") > 0)
    removed = initial_count - df.count()
    print(f"Step 1 - Removed {removed:,} zero amount transactions")

    # Step 2: Add balance discrepancy flag
    df = df.withColumn(
        "is_balance_discrepancy",
        F.when(
            (F.col("type").isin(["TRANSFER", "CASH_OUT"]))
            & (F.col("oldbalanceOrg") > 0)
            & (
                F.round(F.col("oldbalanceOrg") - F.col("amount"), 2)
                != F.round(F.col("newbalanceOrig"), 2)
            ),
            True,
        ).otherwise(False),
    )
    discrepancies = df.filter(F.col("is_balance_discrepancy")).count()
    print(f"Step 2 - Balance discrepancies flagged: {discrepancies:,}")

    # Step 3: Convert step to simulation time fields
    df = df.withColumn("hour_of_simulation", F.col("step").cast(IntegerType()))
    df = df.withColumn("day_of_simulation", (F.col("step") / 24).cast(IntegerType()) + 1)
    print("Step 3 - Added hour_of_simulation and day_of_simulation")

    # Step 4: Encode transaction type
    df = df.withColumn(
        "type_encoded",
        F.when(F.col("type") == "TRANSFER", 0)
        .when(F.col("type") == "CASH_OUT", 1)
        .when(F.col("type") == "PAYMENT", 2)
        .when(F.col("type") == "DEBIT", 3)
        .when(F.col("type") == "CASH_IN", 4)
        .otherwise(-1),
    )
    print("Step 4 - Transaction type encoded")

    # Step 5: Add risk flag
    mean_amount = df.agg(F.mean("amount")).collect()[0][0]
    df = df.withColumn(
        "risk_flag",
        F.when(
            (F.col("type").isin(["TRANSFER", "CASH_OUT"]))
            & (F.col("amount") > mean_amount),
            True,
        ).otherwise(False),
    )
    risk_flagged = df.filter(F.col("risk_flag")).count()
    print(f"Step 5 - Risk flagged transactions: {risk_flagged:,}")

    # Step 6: Add balance difference column
    df = df.withColumn(
        "balance_diff",
        F.col("oldbalanceOrg") - F.col("newbalanceOrig"),
    )
    print("Step 6 - Added balance_diff column")

    # Step 7: Deduplicate
    before_dedup = df.count()
    df = df.dropDuplicates(["nameOrig", "step", "amount", "type"])
    after_dedup = df.count()
    print(f"Step 7 - Deduplicated transactions: {before_dedup - after_dedup:,} duplicates")

    # Step 8: Add pipeline metadata
    df = df.withColumn("silver_processed_at", F.lit(INGESTION_TIMESTAMP))
    df = df.withColumn("pipeline_version", F.lit(PIPELINE_VERSION))
    print("Step 8 - Added pipeline metadata")

    print("\nSilver transformations complete")
    print(f"Final record count: {df.count():,}")
    print(f"Final columns: {len(df.columns)}")

    return df


silver_df = apply_silver_transformations(bronze_df)
silver_df.printSchema()

In [ ]:
def validate_silver_data(df):
    """
    Basic validation checks before writing to Silver layer.
    Catches issues before bad data reaches Gold layer.
    """
    print("=" *60)
    print("SILVER DATA VALIDATION")
    print("="*60)

    checks_passed = 0
    checks_failed = 0
    # Check 1: No zero amounts
    zero_amounts = df.filter(F.col("amount")<=0).count()
    if zero_amounts == 0:
        print(" Check 1 PASSED - No zero amount transactions")
        checks_passed +=1
    else:
        print(f" check 1 Failed - {zero_amounts} zero amounts found")
        checks_failed +=1
    # Check 2: isFraud only 0 or 1
    invalid_fraud = df.filter( ~F.col("isFraud").isin([0,1])).count()
    if invalid_fraud == 0:
        print(" Check 2 PASSED - isFraud values valid")
        checks_passed+=1
    else:
        print(f" Check 2 FAILED — {invalid_fraud} invalid fraud values")
        checks_failed += 1
    #Check 3 - No null amounts
    null_amounts = df.filter(F.col("amount").isNull()).count()
    if null_amounts == 0:
        print(" Check 3 PASSED - No null amounts")
        checks_passed+=1
    else:
        print(f" Check 3 Failed - {null_amounts} null amounts found")
        checks_failed +=1
    
    # Check 4 - Balance discrepancy fraud correlation
    discrepancy_fraud_rate = df.filter(F.col("is_balance_discrepancy")==True).agg(F.mean("isFraud")).collect()[0][0]
    print(f" Check 4 INFO — Fraud rate in discrepancy records: {discrepancy_fraud_rate:.2%}")
    checks_passed+=1
    print(f"\nValidation complete — "
          f"{checks_passed} passed, {checks_failed} failed")
    
    if checks_failed > 0:
        raise Exception(f" {checks_failed} validation checks failed. "
                        f"Pipeline stopped.")
     
    return True

validate_silver_data(silver_df)








In [ ]:
def write_silver_layer(df, output_path):
    """
    Writes validated Silver data to partitioned Parquet.
    Fully distributed PySpark writer — scalable and cloud ready.
    """
    print("=" * 60)
    print("WRITING SILVER LAYER")
    print("=" * 60)

    partition_date = datetime.now().strftime("%Y-%m-%d")
    partition_path = os.path.join(output_path, f"date={partition_date}")

    # Native PySpark distributed Parquet write - prevents Out-Of-Memory errors on scale
    df.write.mode("overwrite").parquet(partition_path)

    print(f"✅ Silver data written to: {partition_path}")
    print(f"Records written: {df.count():,}")
    print(f"Columns: {len(df.columns)}")

    return partition_path

output = write_silver_layer(silver_df, SILVER_OUTPUT_PATH)

In [ ]:
print("=" * 60)
print("FINPULSE SILVER LAYER — COMPLETE SUMMARY")
print("=" * 60)
print(f"Source: Bronze transactions")
print(f"Records processed: {silver_df.count():,}")
print(f"Columns: {len(silver_df.columns)}")
print(f"New columns added:")
print(f"  - is_balance_discrepancy")
print(f"  - hour_of_simulation")
print(f"  - day_of_simulation")
print(f"  - type_encoded")
print(f"  - risk_flag")
print(f"  - balance_diff")
print(f"  - silver_processed_at")
print(f"  - pipeline_version")
print(f"Output: {output}")
print("=" * 60)
print("✅ SILVER LAYER COMPLETE — Ready for Soda Core checks")
print("=" * 60)

spark.stop()